### D603 Task 1: Classification Models Report
**Crystal Ford**



#### B: Data mining
Can we predict whether a customer will churn based on their service usage and demographic features?

The goal is to build a model that helps spot customers who might leave soon. If we can predict that, the company can step in early with things like discounts or special offers to try and keep them.

This is totally doable because we already have data that shows which past customers left and which stayed — so we can teach the model what churn looks like and use that to make future predictions.


#### C: Reasoning and Libraries used
Random Forest is an ensemble learning algorithm that builds multiple decision trees using random subsets of the data and features. Each decision tree independently predicts the outcome. The final prediction is made using **majority voting**—the class predicted by the most trees becomes the model’s final output.

**Key concepts:**
- **Bootstrapping:** Every tree gets a slightly different version of the data to learn from.
- **Random feature selection:** Trees don’t look at all features at once — they each focus on a few, which helps reduce overfitting.
- **Majority voting:** Each tree casts a vote, and the option with the most votes wins.

This method works really well for large datasets with lots of features, and it’s more reliable than using a single decision tree.

#### Python Libraries Used

| Library | Purpose |
|--------|---------|
| `pandas`, `numpy` | Data loading, transformation, numerical operations |
| `sklearn.model_selection` | `train_test_split`, `GridSearchCV` for data splitting and tuning |
| `sklearn.ensemble` | `RandomForestClassifier` to train the model |
| `sklearn.metrics` | Evaluation metrics like accuracy, precision, recall, F1, AUC-ROC |
| `matplotlib`, `seaborn` | Visualizing confusion matrices |


#### D. Data Preparation

Convert all features into numerical format and remove irrelevant columns that do not aid prediction.

All remaining columns after cleaning were used.

- **Categorical (One-hot encoded):** Contract, InternetService, etc.
- **Binary (mapped):** OnlineSecurity, Techie, Churn, etc.
- **Continuous:** MonthlyCharge, Tenure, Population


In [ ]:
# D3: Data Cleaning
import pandas as pd
import numpy as np

df = pd.read_csv("churn_clean.csv")

# Drop irrelevant columns
columns_to_drop = ['CaseOrder', 'Customer_id', 'Interaction', 'UID', 'City', 'State', 'County', 'Zip', 'Lat', 'Lng']
df.drop(columns=columns_to_drop, inplace=True)

# Binary yes/no mapping
binary_map = {'Yes': 1, 'No': 0}
binary_columns = [
    'Techie', 'Port_modem', 'Tablet', 'Phone', 'Multiple',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
    'PaperlessBilling', 'Churn'
]
df[binary_columns] = df[binary_columns].applymap(lambda x: binary_map.get(x, x))

# One-hot encode remaining categoricals
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
non_binary_cats = list(set(categorical_cols) - set(binary_columns))
df = pd.get_dummies(df, columns=non_binary_cats, drop_first=False)

# Confirm all numeric
assert df.select_dtypes(include=['object']).empty, "Non-numeric columns still exist!"

df.to_csv("churn_cleaned.csv", index=False)
print("Cleaned and encoded dataset saved.")


Saved as `churn_cleaned.csv`


#### E: Data Analysis


In [ ]:
# E1: Split the data
from sklearn.model_selection import train_test_split

df_clean = pd.read_csv("churn_cleaned.csv")
X = df_clean.drop('Churn', axis=1)
y = df_clean['Churn']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

X_train.to_csv("X_train.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
X_val.to_csv("X_val.csv", index=False)
y_val.to_csv("y_val.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Train, validation, and test sets saved.")


In [ ]:
# E2: Initial Model Training & Validation
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

y_val_pred = rf_model.predict(X_val)
y_val_proba = rf_model.predict_proba(X_val)[:, 1]

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Validation Precision:", precision_score(y_val, y_val_pred))
print("Validation Recall:", recall_score(y_val, y_val_pred))
print("Validation F1 Score:", f1_score(y_val, y_val_pred))
print("Validation AUC-ROC:", roc_auc_score(y_val, y_val_proba))

cm = confusion_matrix(y_val, y_val_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
plt.title("Validation Confusion Matrix")
plt.show()


In [ ]:
# E3: Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

print("Best Parameters Found:", grid_search.best_params_)
print("Best F1 Score from GridSearchCV:", grid_search.best_score_)


In [ ]:
# E4: Evaluate Best Model on Test Set
best_model = grid_search.best_estimator_
y_test_pred = best_model.predict(X_test)
y_test_proba = best_model.predict_proba(X_test)[:, 1]

print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("Test Precision:", precision_score(y_test, y_test_pred))
print("Test Recall:", recall_score(y_test, y_test_pred))
print("Test F1 Score:", f1_score(y_test, y_test_pred))
print("Test AUC-ROC:", roc_auc_score(y_test, y_test_proba))

cm_test = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens', xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
plt.title("Test Confusion Matrix")
plt.show()


#### F: Results Summary and Recommendation

#### F1: Comparing the Two Models

| What We Measured | First Try (Validation) | Final Version (Test Set) |
|------------------|-------------------------|---------------------------|
| **Accuracy** – How often the model was right overall | 88.6% | 87.8% |
| **Precision** – Of those predicted to leave, how many actually did? | 88.3% | 86.3% |
| **Recall** – Of all who really left, how many did the model catch? | 65.5% | 63.9% |
| **F1 Score** – A balance between precision and recall | 75.2% | 73.5% |
| **AUC-ROC** – How well it separates leavers from stayers | 95.7% | 95.1% |

**What this means:**
The final model performed almost the same as the first one, which means the model is stable and reliable. It's especially strong at spotting people who might leave, and it's really good at separating churners from non-churners.

####  F2: What the Results Mean

This model is a great tool for finding customers who are at high risk of leaving. It's especially good at being careful with its predictions — meaning it usually only flags people as churn risks when they really are. This could be super useful for helping the business reduce customer loss.

#### F3: One Limitation

Right now, the model only uses basic customer info and service data. It doesn’t include recent behavior or how satisfied the customer might be. Also, if the company doesn’t retrain the model regularly with new data, it might get less accurate over time.

#### F4: Recommendation

Use this model to check which customers are most likely to leave. Then reach out to those high-risk customers with special deals, personal support, or loyalty rewards. That way, the company can keep more customers and reduce churn. It would also be beneficial to have surveys or feedback from customers about how likely they are to churn or what would entice them to do so.



#### H: Third-Party Code Sources
- Scikit-learn Documentation: https://scikit-learn.org/
- Pandas Documentation: https://pandas.pydata.org/
- Stack Overflow syntax snippets (e.g., for GridSearchCV)


#### I: References
WGU course materials

Scikit-learn developers. (n.d.). *Scikit-learn: Machine Learning in Python*. Retrieved from https://scikit-learn.org/

